### Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import tensorflow.keras as keras
import matplotlib.pyplot as plt
from numpy.polynomial.polynomial import polyfit

### Import the data as "df"

In [2]:
df = pd.read_csv('data_csv.csv')

### Functions for normalizing "time" and "attempts" columns

In [3]:
def normalize_time(time):
    """
    To normalize time, any time below 5 minutes (5 * 60 = 300 seconds) will be divided by 300; any time above 5 minutes will be translated to 1.
    """

    if time >= 300:
        return 1.0
    else:
        return time / 300
    

def normalize_attempts(attempts):
    """
    The data is collected from tests, each having 10 questions, each question having 4 choices. So, the minimum possible attempts for each test is 10, while the maximum is 40.
    Using this knowledge, we can normalize the "attempts" in the data frame by:
    """

    minimum = 10
    maximum = 40

    return (attempts - minimum)/(maximum - minimum)

    "This ensures that a test answered by 10 attempts is translated to 0, while a test answered by 40 to 1."

df["time"] = df["time"].apply(normalize_time)
df["attempts"] = df["attempts"].apply(normalize_attempts)

### Seeing the first 5 values in the dataframe

In [ ]:
df.head()

### Preparing data for the first neural network

In [5]:
x_accuracy = list()
y_accuracy = list()

for i in range(df.shape[0]):
    x_accuracy.append(df["accuracy"][i])
    y_accuracy.append(df["next_difficulty"][i])

x_accuracy = np.array(x_accuracy)
y_accuracy = np.array(y_accuracy)

### Simple way to split data into training and testing

In [6]:
x_accuracy_train, y_accuracy_train, x_accuracy_test, y_accuracy_test = x_accuracy[:int(0.8 * len((x_accuracy)))], y_accuracy[:int(0.8 * len((y_accuracy)))], x_accuracy[int(0.8 * len((x_accuracy))):], y_accuracy[int(0.8 * len((y_accuracy))):]

### Categorizing the outputs

In [7]:
y_accuracy_train, y_accuracy_test = keras.utils.to_categorical(y_accuracy_train, 3), keras.utils.to_categorical(y_accuracy_test, 3)

### Defining the model

In [ ]:
model_difficulty_predictor = Sequential()
model_difficulty_predictor.add(Dense(units = 256, activation = "relu", input_shape = (1,)))
model_difficulty_predictor.add(Dense(units = 128, activation = "relu"))
model_difficulty_predictor.add(Dense(units = 3, activation = "sigmoid"))

### Model summary

In [ ]:
model_difficulty_predictor.summary()

### Model compiling

In [10]:
model_difficulty_predictor.compile(loss = "categorical_crossentropy", metrics = ["accuracy"])

### Model training

In [ ]:
nb_of_epochs = 100
history = model_difficulty_predictor.fit(x_accuracy_train, y_accuracy_train, epochs = nb_of_epochs, verbose = 1, validation_data = (x_accuracy_test, y_accuracy_test))

### Visualization of the model's progress

In [ ]:
chart_x = range(1,nb_of_epochs + 1)
chart_y_train = history.history['accuracy']
chart_y_test = history.history['val_accuracy']
print(history.history.keys())

def plot_learning():
    plt.plot(chart_x, chart_y_train, "r-", label = "training accurcay")
    plt.plot(chart_x, chart_y_test, "b-", label = "validation accuracy")
    plt.xlabel("Training epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

plot_learning()

### Preparing data for the grade prediction

In [13]:
y_grades = list()

for i in range(df.shape[0]):
    y_grades.append(df["grade"][i])
y_grades = np.array(y_grades)


### Linear regression to predict "grade"

In [ ]:
b_accuracy, m_accuracy = polyfit(x_accuracy, y_grades, 1)
y_hat_accuracy = x_accuracy * m_accuracy + b_accuracy
plt.scatter(x_accuracy, y_grades)

plt.plot(x_accuracy, y_hat_accuracy, "r-")

### Preparing data for second neural network

In [15]:
x_improvement = list()
y_improvement = list()
for i in range(df.shape[0]):
    x_improvement.append([df["accuracy"][i], df["time"][i], df["attempts"][i]])
    y_improvement.append([df["improvement"][i]])

x_improvement = np.array(x_improvement)
y_improvement = np.array(y_improvement)

### Splitting data into training and testing

In [16]:
x_improvement_train, x_improvement_test = x_improvement[:int(0.8*len(x_improvement))], x_improvement[int(0.8*len(x_improvement)):]
y_improvement_train, y_improvement_test = y_improvement[:int(0.8*len(y_improvement))], y_improvement[int(0.8*len(y_improvement)):]

### Defining the model

In [17]:
model_improvement = Sequential()
model_improvement.add(Dense(units = 256, activation = "relu", input_shape = (3,)))
model_improvement.add(Dense(units = 128, activation = "relu"))
model_improvement.add(Dense(units = 3, activation = "sigmoid"))

### Categorizing data, compiling the model and training it

In [ ]:
y_improvement_train, y_improvement_test = keras.utils.to_categorical(y_improvement_train, 3), keras.utils.to_categorical(y_improvement_test, 3)

model_improvement.summary()

model_improvement.compile(loss = "categorical_crossentropy", metrics = ["accuracy"])

nb_of_epochs_improvement = 200
history_improvement = model_improvement.fit(x_improvement_train, y_improvement_train, epochs = nb_of_epochs_improvement, verbose = 1, validation_data = (x_improvement_test, y_improvement_test))

### Visualization of the model's progress

In [ ]:
chart_x = range(1, nb_of_epochs_improvement + 1)
chart_y_train = history_improvement.history['accuracy']
chart_y_test = history_improvement.history['val_accuracy']
print(history_improvement.history.keys())

def plot_learning():
    plt.plot(chart_x, chart_y_train, "r-", label = "training accurcay")
    plt.plot(chart_x, chart_y_test, "b-", label = "validation accuracy")
    plt.xlabel("Training epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

plot_learning()